Install Libraries


In [1]:
!pip install pandas scikit-learn matplotlib seaborn

Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import classification_report, confusion_matrix

Load Dataset

In [4]:
df = pd.read_csv("/content/SMSSpamCollection", sep='\t', names=['label', 'message'])
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Explore Data

In [5]:
print("Shape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
print("\nClass Distribution:\n", df['label'].value_counts())

Shape: (5572, 2)

Null values:
 label      0
message    0
dtype: int64

Class Distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64


Convert Labels

In [6]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


Text Preprocessing

In [7]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    return text

df['message'] = df['message'].apply(preprocess)
df.head()

,label,message
0,0,go until jurong point crazy available only in ...
1,0,ok lar joking wif u oni
2,1,free entry in a wkly comp to win fa cup final...
3,0,u dun say so early hor u c already then say
4,0,nah i dont think he goes to usf he lives aroun...


Train-Test Split

In [8]:
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TF-IDF Vectorization

In [9]:
tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Train Naive Bayes Model

In [10]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

MultinomialNB()

Predictions

In [11]:
y_pred = model.predict(X_test_tfidf)

Evaluation

In [12]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[966   0]
 [ 35 114]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



Prediction Function

In [15]:
def predict_message(msg):
    msg = preprocess(msg)
    vec = tfidf.transform([msg])
    result = model.predict(vec)[0]

    if result == 1:
        return "Spam"
    else:
        return "not spam"

Test Custom Messages

In [16]:
print(predict_message("Congratulations! You won a free iPhone"))
print(predict_message("Let's meet at 5 pm"))

Spam
not spam


Logistic Regression

In [17]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("Logistic Regression:\n")
print(classification_report(y_test, y_pred_lr))

Logistic Regression:

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.74      0.85       149

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.97      0.96      1115



Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train_tfidf, y_train)

y_pred_rf = rf.predict(X_test_tfidf)

print("Random Forest:\n")
print(classification_report(y_test, y_pred_rf))

Random Forest:

              precision    recall  f1-score   support

           0       0.97      1.00      0.99       966
           1       1.00      0.81      0.90       149

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115



Handle Imbalance (Class Weight)

In [19]:
lr_balanced = LogisticRegression(class_weight='balanced')
lr_balanced.fit(X_train_tfidf, y_train)

y_pred_bal = lr_balanced.predict(X_test_tfidf)

print("Balanced Logistic Regression:\n")
print(classification_report(y_test, y_pred_bal))

Balanced Logistic Regression:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.95      0.93      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

